In [ ]:
# ─── ЯЧЕЙКА 1: Проверка окружения ───────────────────────────────────────────
import torch
import transformers

print('torch:        ', torch.__version__)
print('transformers: ', transformers.__version__)
print('CUDA:         ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:          ', torch.cuda.get_device_name(0))
    print('VRAM free:    ', round(torch.cuda.mem_get_info()[0] / 1024**3, 2), 'GB')


In [ ]:
# ─── ЯЧЕЙКА 2: Загрузка токенайзера ─────────────────────────────────────────
from transformers import AutoTokenizer

MODEL_PATH = r'D:\bogdanov\PyProjects\Agents_project\Models\Qwen2.5-14B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print('Tokenizer loaded:', tokenizer.vocab_size, 'tokens')


In [ ]:
# ─── ЯЧЕЙКА 3: Загрузка модели (4-bit) ──────────────────────────────────────
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map='cuda',
    trust_remote_code=True
)
model.eval()

print('Model loaded on:', next(model.parameters()).device)
print('VRAM used:      ', round(torch.cuda.memory_allocated() / 1024**3, 2), 'GB')


In [10]:
# ─── ЯЧЕЙКА 4: Конфиг, пороговые значения, пути ─────────────────────────────
import json
import os

AGENT_LOGS_PATH = r'D:\bogdanov\PyProjects\Agents_project\agent_logs.json'
REGISTRY_PATH   = r'D:\bogdanov\PyProjects\Agents_project\registry.json'
POLL_INTERVAL_SEC = 5

# ── Пороговые значения метрик (MVP) ──────────────────────────────────────────
THRESHOLDS = {
    'requests_per_min':   {'warn': 30,   'crit': 50,   'unit': 'req/min',  'desc': 'Request rate'},
    'avg_response_ms':    {'warn': 2000, 'crit': 3500, 'unit': 'ms',       'desc': 'Avg response time'},
    'memory_mb':          {'warn': 600,  'crit': 1200, 'unit': 'MB',       'desc': 'Memory usage'},
    'error_rate_pct':     {'warn': 3.0,  'crit': 8.0,  'unit': '%',        'desc': 'Error rate'},
    'queue_depth':        {'warn': 10,   'crit': 20,   'unit': 'tasks',    'desc': 'Queue depth'},
    'tokens_per_request': {'warn': 700,  'crit': 900,  'unit': 'tokens',   'desc': 'Tokens per request'},
    'uptime_pct':         {'warn': 99.0, 'crit': 97.0, 'unit': '%',        'desc': 'Uptime',  'inverted': True},
}

def load_registry(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_agent_logs(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

print('Config loaded')
print('Metrics monitored:', list(THRESHOLDS.keys()))
print('Poll interval:    ', POLL_INTERVAL_SEC, 'sec')


Config loaded
Metrics monitored: ['requests_per_min', 'avg_response_ms', 'memory_mb', 'error_rate_pct', 'queue_depth', 'tokens_per_request', 'uptime_pct']
Poll interval:     5 sec


In [11]:
# ─── ЯЧЕЙКА 5: Проверка метрик + формирование отчёта ────────────────────────
from datetime import datetime

LEVEL_OK   = 'OK'
LEVEL_WARN = 'WARN'
LEVEL_CRIT = 'CRIT'


def check_metric(name, value, threshold):
    inverted = threshold.get('inverted', False)
    warn = threshold['warn']
    crit = threshold['crit']
    if not inverted:
        if value >= crit:
            return LEVEL_CRIT
        elif value >= warn:
            return LEVEL_WARN
        return LEVEL_OK
    else:
        if value <= crit:
            return LEVEL_CRIT
        elif value <= warn:
            return LEVEL_WARN
        return LEVEL_OK


def delta_str(value, threshold):
    inverted = threshold.get('inverted', False)
    ref = threshold['warn']
    if not inverted:
        diff = value - ref
        return '+' + str(round(diff, 2)) + ' over WARN'
    else:
        diff = ref - value
        return '-' + str(round(diff, 2)) + ' below WARN'


def check_all_agents(agent_logs):
    report = {
        'timestamp':  agent_logs.get('timestamp', str(datetime.now())),
        'checked_at': str(datetime.now()),
        'agents':     [],
        'violations': [],
        'has_issues': False
    }

    for agent in agent_logs.get('agents', []):
        name    = agent['name']
        metrics = agent.get('metrics', {})
        agent_result = {'name': name, 'metrics': [], 'max_level': LEVEL_OK}

        for metric_name, threshold in THRESHOLDS.items():
            if metric_name not in metrics:
                continue
            value = metrics[metric_name]
            level = check_metric(metric_name, value, threshold)

            entry = {
                'metric':    metric_name,
                'value':     value,
                'unit':      threshold['unit'],
                'desc':      threshold['desc'],
                'warn':      threshold['warn'],
                'crit':      threshold['crit'],
                'level':     level,
                'delta':     delta_str(value, threshold) if level != LEVEL_OK else ''
            }
            agent_result['metrics'].append(entry)

            if level != LEVEL_OK:
                report['has_issues'] = True
                report['violations'].append({
                    'agent':  name,
                    'metric': metric_name,
                    'value':  value,
                    'level':  level,
                    'delta':  entry['delta']
                })
                if level == LEVEL_CRIT:
                    agent_result['max_level'] = LEVEL_CRIT
                elif agent_result['max_level'] != LEVEL_CRIT:
                    agent_result['max_level'] = LEVEL_WARN

        report['agents'].append(agent_result)

    return report


def print_report(report):
    print('\n' + '=' * 64)
    print('METRICS REPORT  |  ' + report['checked_at'])
    print('Snapshot time:  |  ' + report['timestamp'])
    print('=' * 64)

    icons = {LEVEL_OK: 'OK  ', LEVEL_WARN: 'WARN', LEVEL_CRIT: 'CRIT'}

    for agent in report['agents']:
        lvl = agent['max_level']
        print('\n[' + icons[lvl] + '] ' + agent['name'])
        for m in agent['metrics']:
            icon = icons[m['level']]
            line = '  ' + icon + '  ' + m['desc'].ljust(22)
            line += str(m['value']).rjust(8) + ' ' + m['unit'].ljust(8)
            if m['delta']:
                line += '  <- ' + m['delta']
            print(line)

    print('\n' + '-' * 64)
    if report['has_issues']:
        print('VIOLATIONS: ' + str(len(report['violations'])))
        for v in report['violations']:
            print('  [' + v['level'] + '] ' + v['agent'] + ' / ' + v['metric'] + ' = ' + str(v['value']) + '  ' + v['delta'])
    else:
        print('All agents nominal.')
    print('=' * 64)


print('check_all_agents() ready')
print('print_report()     ready')


check_all_agents() ready
print_report()     ready


In [12]:
# ─── ЯЧЕЙКА 6: Промпт + LLM → JSON команды движку ─────────────────────────
import torch
import json
import re


def build_logger_system_prompt(registry):
    agents_block = '\n'.join([
        '  - ' + a['name'] + ': ' + a['system_prompt']
        for a in registry['agents']
    ])
    return (
        'You are the system Logger of a multi-agent AI ecosystem.\n'
        'You receive a metrics snapshot with highlighted violations.\n'
        '\n'
        'Your ONLY job is to output a JSON array of commands for the engine.\n'
        'Do NOT explain. Do NOT recommend. Do NOT output text.\n'
        'Output ONLY a valid JSON array.\n'
        '\n'
        'ALLOWED ACTIONS:\n'
        '  edit      — change agent config (throttling, memory limit)\n'
        '  split     — split overloaded agent into two\n'
        '  add       — add new agent if queue is critically overloaded\n'
        '  remove    — remove agent with zero utilization\n'
        '  no_action — if violations are minor and no structural change needed\n'
        '\n'
        'DECISION RULES:\n'
        '  CRIT error_rate or avg_response_ms  -> edit (reduce load) or split\n'
        '  CRIT memory_mb                      -> edit (set memory limit)\n'
        '  CRIT queue_depth + CRIT requests    -> split or add\n'
        '  WARN only                           -> edit with soft limits\n'
        '  all OK                              -> no_action\n'
        '\n'
        'OUTPUT FORMAT (strict, no markdown, no explanation):\n'
        '[\n'
        '  {"action": "edit", "agent": "AgentName", "target": "config", "data": {"max_requests_per_min": 30}},\n'
        '  {"action": "split", "source_agent": "AgentName", "target_agents": ["AgentA", "AgentB"]}\n'
        ']\n'
        '\n'
        'KNOWN AGENTS:\n'
        + agents_block
    )


def build_logger_user_prompt(report):
    lines = []
    lines.append('METRICS SNAPSHOT: ' + report['timestamp'])
    lines.append('CHECKED AT: ' + report['checked_at'])
    lines.append('')
    for agent in report['agents']:
        lvl = agent['max_level']
        lines.append('[' + lvl + '] ' + agent['name'])
        for m in agent['metrics']:
            flag = ' *** ' + m['level'] + ' *** ' if m['level'] != 'OK' else '       '
            line = '  ' + flag + m['desc'] + ': ' + str(m['value']) + ' ' + m['unit']
            if m['delta']:
                line += ' (' + m['delta'] + ')'
            lines.append(line)
        lines.append('')
    lines.append('VIOLATION SUMMARY (' + str(len(report['violations'])) + ' total):')
    for v in report['violations']:
        lines.append('  [' + v['level'] + '] ' + v['agent'] + ' -> ' + v['metric'] + ' = ' + str(v['value']) + '  ' + v['delta'])
    return '\n'.join(lines)


def call_llm_logger(report, registry):
    system_prompt = build_logger_system_prompt(registry)
    user_prompt   = build_logger_user_prompt(report)

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt')
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            use_cache=True,
        )

    raw = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if match:
        raw = match.group(0)

    try:
        commands = json.loads(raw)
        if not isinstance(commands, list):
            commands = [commands]
    except json.JSONDecodeError:
        print('[WARN] JSON parse failed, raw output:')
        print(raw[:300])
        commands = []

    return commands


print('build_logger_system_prompt() ready')
print('call_llm_logger()            ready')


build_logger_system_prompt() ready
call_llm_logger()            ready


In [13]:
# ─── ЯЧЕЙКА 7: Главный цикл мониторинга ────────────────────────────────────
import time
import json
from datetime import datetime

registry = load_registry(REGISTRY_PATH)

print('Logger started')
print('Monitoring:', AGENT_LOGS_PATH)
print('Interval:  ', POLL_INTERVAL_SEC, 'sec')
print('Press Ctrl+C to stop\n')

cycle = 0

try:
    while True:
        cycle += 1
        ts = datetime.now().strftime('%H:%M:%S')
        print('[' + ts + '] Cycle #' + str(cycle) + ' — reading metrics...')

        try:
            agent_logs = load_agent_logs(AGENT_LOGS_PATH)
        except Exception as e:
            print('[ERROR] Cannot read agent_logs.json:', e)
            time.sleep(POLL_INTERVAL_SEC)
            continue

        report = check_all_agents(agent_logs)
        print_report(report)

        if report['has_issues']:
            crit_count = sum(1 for v in report['violations'] if v['level'] == 'CRIT')
            warn_count = sum(1 for v in report['violations'] if v['level'] == 'WARN')
            print('[LLM] Violations: ' + str(crit_count) + ' CRIT, ' + str(warn_count) + ' WARN — generating commands...')

            commands = call_llm_logger(report, registry)

            if commands:
                print('\n' + '=' * 64)
                print('ENGINE COMMANDS')
                print('=' * 64)
                print(json.dumps(commands, ensure_ascii=False, indent=2))
                print('=' * 64)
            else:
                print('[LLM] No valid commands generated')
        else:
            print('[cycle skipped, all ok!]')

        print()
        time.sleep(POLL_INTERVAL_SEC)

except KeyboardInterrupt:
    print('\nLogger stopped.')


Logger started
Monitoring: D:\bogdanov\PyProjects\Agents_project\agent_logs.json
Interval:   5 sec
Press Ctrl+C to stop

[18:34:03] Cycle #1 — reading metrics...

METRICS REPORT  |  2026-06-11 18:34:03.355503
Snapshot time:  |  2025-10-01T11:00:00

[OK  ] FinanceAgent
  OK    Request rate                10 req/min 
  OK    Avg response time          800 ms      
  OK    Memory usage               200 MB      
  OK    Error rate                 0.5 %       
  OK    Queue depth                  2 tasks   
  OK    Tokens per request         300 tokens  
  OK    Uptime                    99.9 %       

[WARN] ResearchAgent
  WARN  Request rate                35 req/min   <- +5 over WARN
  WARN  Avg response time         2400 ms        <- +400 over WARN
  OK    Memory usage               350 MB      
  OK    Error rate                 1.0 %       
  OK    Queue depth                  4 tasks   
  OK    Tokens per request         450 tokens  
  OK    Uptime                    99.8 %       

